# 01 Streaming 2.5D Cache Builder

Inputs:

- output dataset from notebook `00_archive_listing_manifest.ipynb`
- the three raw no-autoextract ViMed-PET Kaggle datasets

This notebook creates a memory-safe 2.5D cache without extracting an
entire source base. It extracts CT/PET files in small batches, converts
each case into six `uint8` channels, flushes the memmap cache, and removes
temporary raw arrays after every batch.

The notebook is resumable. If Kaggle disconnects, rerun with the previous
output attached and already cached case ids will be skipped.


In [ ]:
from pathlib import Path
import os, re, json, shutil, subprocess, warnings, time
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

OUTPUT_ROOT = Path("/kaggle/working/vimedpet_q1_outputs") if Path("/kaggle/working").exists() else Path("vimedpet_q1_outputs")
CACHE_DIR = OUTPUT_ROOT / "01_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
SCRATCH_ROOT = Path("/kaggle/temp/vimedpet_q1_cache_stream") if Path("/kaggle").exists() else OUTPUT_ROOT / "_scratch_cache_stream"
STAGE_ROOT = SCRATCH_ROOT / "archive_stage"
BATCH_ROOT = SCRATCH_ROOT / "batch_extract"

for p in [SCRATCH_ROOT, STAGE_ROOT, BATCH_ROOT]:
    if p.exists():
        shutil.rmtree(p, ignore_errors=True)
    p.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 224
CHANNELS_25D = 6
BATCH_CASES_PER_EXTRACT = 6
# Use None for full cache. For a quick smoke run, set for example 120.
CACHE_CASE_LIMIT = None
# Use None for all splits. To validate quickly first: ["validation", "test"].
CACHE_ONLY_SPLITS = None
RESUME_FROM_EXISTING = True

INPUT_HINTS = [
    "/kaggle/input/datasets/tundng111/vimed-pet-ct-part1-raw-no-autoextract-v3",
    "/kaggle/input/datasets/tundng111/vimed-pet-ct-part2-raw-no-autoextract-v1",
    "/kaggle/input/datasets/tundng111/vimed-pet-ct-part3-raw-no-autoextract-v1",
    "/kaggle/input",
    str(Path.cwd()),
]

def show_disk_usage(label):
    print(f"\nDisk usage: {label}")
    for raw_path in ["/kaggle/temp", "/tmp", "/kaggle/working"]:
        p = Path(raw_path)
        if p.exists():
            usage = shutil.disk_usage(p)
            print(raw_path, "free GiB:", round(usage.free / 1024**3, 2), "used GiB:", round(usage.used / 1024**3, 2), "total GiB:", round(usage.total / 1024**3, 2))

print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("CACHE_DIR:", CACHE_DIR)
print("SCRATCH_ROOT:", SCRATCH_ROOT)
show_disk_usage("after setup")

def import_previous_cache_if_available():
    if not Path("/kaggle/input").exists():
        return
    if (CACHE_DIR / "q1_cache_meta.json").exists() and (CACHE_DIR / "q1_cache_index.csv").exists():
        return
    candidates = sorted(Path("/kaggle/input").rglob("q1_cache_meta.json"), key=lambda p: len(str(p)))
    for meta_path in candidates:
        src_dir = meta_path.parent
        try:
            meta = json.loads(meta_path.read_text(encoding="utf-8"))
        except Exception:
            continue
        mmap_name = meta.get("mmap_path")
        required = [src_dir / "q1_cache_index.csv", src_dir / "q1_clean_manifest_for_cache.csv", src_dir / str(mmap_name)]
        if not mmap_name or not all(p.exists() for p in required):
            continue
        print("Importing previous cache from:", src_dir)
        for p in required + [meta_path]:
            dst = CACHE_DIR / p.name
            if not dst.exists():
                shutil.copy2(p, dst)
        return

import_previous_cache_if_available()


In [ ]:
def locate_clean_manifest():
    roots = []
    if Path("/kaggle/input").exists():
        roots += list(Path("/kaggle/input").rglob("q1_clean_manifest.csv"))
    roots += list(Path.cwd().rglob("q1_clean_manifest.csv"))
    roots = [p for p in roots if "01_cache" not in str(p)]
    if not roots:
        raise FileNotFoundError("q1_clean_manifest.csv not found. Attach notebook 00 output as input.")
    return sorted(roots, key=lambda p: len(str(p)))[0]

def clean_value(x):
    s = "" if pd.isna(x) else str(x).strip()
    if s.lower() in {"nan", "none", "null"}:
        return ""
    return re.sub(r"\s+", " ", s)

def build_model_target(row):
    findings = clean_value(row.get("findings", ""))
    impression = clean_value(row.get("impression", ""))
    parts = []
    if findings:
        parts.append("FINDINGS:\n" + findings)
    if impression:
        parts.append("IMPRESSION:\n" + impression)
    target = "\n\n".join(parts).strip()
    if not target:
        target = clean_value(row.get("report_text_clean", row.get("target_text", "")))
    return target

manifest_path = locate_clean_manifest()
clean = pd.read_csv(manifest_path)
clean = clean.reset_index(drop=True)
clean["row_index"] = np.arange(len(clean), dtype=np.int64)
if CACHE_ONLY_SPLITS is not None:
    clean = clean[clean["clean_split"].isin(CACHE_ONLY_SPLITS)].copy()
if CACHE_CASE_LIMIT is not None:
    clean = clean.head(CACHE_CASE_LIMIT).copy()

clean["model_target_text"] = clean.apply(build_model_target, axis=1)
clean["target_text"] = clean["model_target_text"]
section_check = pd.DataFrame([
    {"field": "findings", "present": int(clean["findings"].fillna("").astype(str).str.strip().ne("").sum())},
    {"field": "impression", "present": int(clean["impression"].fillna("").astype(str).str.strip().ne("").sum())},
    {"field": "model_target_has_impression", "present": int(clean["model_target_text"].str.contains("IMPRESSION:", regex=False).sum())},
])
section_check["total"] = int(len(clean))
section_check["coverage"] = section_check["present"] / section_check["total"].clip(lower=1)
section_check["coverage_percent"] = (section_check["coverage"] * 100).round(2)

print("Manifest:", manifest_path)
print("Rows selected:", clean.shape)
display(clean.groupby("clean_split").size().rename("cases").reset_index())
display(clean.groupby("source_part").size().rename("cases").reset_index())
display(Markdown("### Target section coverage used for cache/modeling"))
display(section_check)
assert float(section_check.loc[section_check.field == "impression", "coverage"].iloc[0]) >= 0.90, section_check
assert float(section_check.loc[section_check.field == "model_target_has_impression", "coverage"].iloc[0]) >= 0.90, section_check


In [ ]:
def existing_roots():
    roots = []
    for x in INPUT_HINTS:
        p = Path(x)
        if p.exists():
            roots.append(p)
    seen, out = set(), []
    for r in roots:
        rr = r.resolve()
        if rr not in seen:
            out.append(rr)
            seen.add(rr)
    return out

def sevenzip_binary():
    for exe in ["7z", "7zz", "7za"]:
        try:
            subprocess.run([exe], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            return exe
        except Exception:
            continue
    raise RuntimeError("7z/7zz/7za not found")

def find_archive_group(source_base):
    hits = []
    for root in existing_roots():
        for marker in list(root.rglob(f"{source_base}.zipraw")) + list(root.rglob(f"{source_base}.zip")):
            parts = sorted(marker.parent.glob(f"{source_base}.z[0-9][0-9]"))
            hits.append({"marker": marker, "parts": parts, "folder": marker.parent})
    if not hits:
        raise FileNotFoundError(f"Archive marker not found for {source_base}")
    return sorted(hits, key=lambda h: len(h["parts"]), reverse=True)[0]

def symlink_required(src, dst):
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    try:
        os.symlink(src, dst)
    except Exception as exc:
        raise RuntimeError("Symlink failed; copying archives is intentionally disabled to protect temp disk.") from exc

def stage_archive(source_base, group):
    stage = STAGE_ROOT / source_base
    if stage.exists():
        shutil.rmtree(stage, ignore_errors=True)
    stage.mkdir(parents=True, exist_ok=True)
    names = [p.name for p in group["parts"]]
    expected = [f"{source_base}.z{i:02d}" for i in range(1, len(group["parts"]) + 1)]
    missing = [x for x in expected if x not in names]
    if missing:
        raise FileNotFoundError(f"Missing split volumes for {source_base}: {missing}")
    for part in group["parts"]:
        symlink_required(part, stage / part.name)
    symlink_required(group["marker"], stage / f"{source_base}.zip")
    print(f"{source_base}: parts={len(group['parts'])}, marker={group['marker'].name}")
    return stage / f"{source_base}.zip"

def run_7z_extract(zip_path, out_dir, archive_paths):
    if out_dir.exists():
        shutil.rmtree(out_dir, ignore_errors=True)
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [sevenzip_binary(), "x", str(zip_path), f"-o{out_dir}", "-y"] + list(archive_paths)
    result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if result.returncode != 0:
        print(result.stdout[-2000:])
        raise RuntimeError("7z batch extraction failed")
    return out_dir


In [ ]:
def resize_u8(x, size=IMAGE_SIZE):
    return np.asarray(Image.fromarray(x).resize((size, size), Image.BILINEAR), dtype=np.uint8)

def normalize_uint8(x, lo=None, hi=None):
    x = np.asarray(x, dtype=np.float32)
    if lo is None:
        lo = float(np.nanpercentile(x, 1))
    if hi is None:
        hi = float(np.nanpercentile(x, 99))
    x = np.clip(x, lo, hi)
    return ((x - lo) / max(hi - lo, 1e-6) * 255).astype(np.uint8)

def sample_slice(vol, frac):
    z = vol.shape[0]
    idx = min(max(int(round((z - 1) * frac)), 0), z - 1)
    return np.asarray(vol[idx])

def make_25d(ct, pet):
    ct35 = resize_u8(normalize_uint8(sample_slice(ct, 0.35), -1000, 1000))
    ct50 = resize_u8(normalize_uint8(sample_slice(ct, 0.50), -1000, 1000))
    ct65 = resize_u8(normalize_uint8(sample_slice(ct, 0.65), -1000, 1000))
    pet_arr = np.asarray(pet)
    pet_ax = resize_u8(normalize_uint8(pet_arr.max(axis=0)))
    pet_cor = resize_u8(normalize_uint8(pet_arr.max(axis=1)))
    pet_sag = resize_u8(normalize_uint8(pet_arr.max(axis=2)))
    return np.stack([ct35, ct50, ct65, pet_ax, pet_cor, pet_sag], axis=0)

def batch_iter(df, n):
    rows = df.to_dict("records")
    for i in range(0, len(rows), n):
        yield rows[i:i+n]


In [ ]:
n_total = int(clean["row_index"].max()) + 1
mmap_path = CACHE_DIR / f"q1_25d_uint8_{IMAGE_SIZE}.memmap"
if mmap_path.exists():
    arr = np.memmap(mmap_path, dtype=np.uint8, mode="r+", shape=(n_total, CHANNELS_25D, IMAGE_SIZE, IMAGE_SIZE))
else:
    arr = np.memmap(mmap_path, dtype=np.uint8, mode="w+", shape=(n_total, CHANNELS_25D, IMAGE_SIZE, IMAGE_SIZE))

cache_index_path = CACHE_DIR / "q1_cache_index.csv"
if RESUME_FROM_EXISTING and cache_index_path.exists():
    cache_index = pd.read_csv(cache_index_path)
    done_cases = set(cache_index.loc[cache_index["cache_ok"] == True, "case_id"].astype(str))
    cache_rows = cache_index.to_dict("records")
else:
    done_cases = set()
    cache_rows = []

selected = clean[~clean["case_id"].astype(str).isin(done_cases)].copy()
print("Already cached:", len(done_cases), "remaining:", len(selected))

start_time = time.time()
source_order = selected["source_part"].drop_duplicates().tolist()
for source_base in source_order:
    group_df = selected[selected["source_part"] == source_base].copy()
    archive_group = find_archive_group(source_base)
    zip_path = stage_archive(source_base, archive_group)
    try:
        for batch_no, batch in enumerate(batch_iter(group_df, BATCH_CASES_PER_EXTRACT), start=1):
            batch_dir = BATCH_ROOT / source_base / f"batch_{batch_no:05d}"
            archive_paths = []
            for row in batch:
                archive_paths.append(row["ct_path"])
                archive_paths.append(row["pet_path"])
            show_disk_usage(f"before {source_base} batch {batch_no}")
            run_7z_extract(zip_path, batch_dir, archive_paths)

            new_rows = []
            for row in batch:
                try:
                    ct_file = batch_dir / row["ct_path"]
                    pet_file = batch_dir / row["pet_path"]
                    ct = np.load(ct_file, mmap_mode="r")
                    pet = np.load(pet_file, mmap_mode="r")
                    arr[int(row["row_index"])] = make_25d(ct, pet)
                    ok, err = True, ""
                    ct_shape, pet_shape = str(tuple(ct.shape)), str(tuple(pet.shape))
                except Exception as exc:
                    ok, err = False, str(exc)
                    ct_shape, pet_shape = "", ""
                rec = {
                    "row_index": int(row["row_index"]),
                    "case_id": row["case_id"],
                    "source_part": source_base,
                    "clean_split": row["clean_split"],
                    "ct_path": row["ct_path"],
                    "pet_path": row["pet_path"],
                    "cache_ok": ok,
                    "cache_error": err,
                    "ct_shape": ct_shape,
                    "pet_shape": pet_shape,
                }
                cache_rows.append(rec)
                new_rows.append(rec)
            arr.flush()
            pd.DataFrame(cache_rows).drop_duplicates("case_id", keep="last").to_csv(cache_index_path, index=False, encoding="utf-8-sig")
            shutil.rmtree(batch_dir, ignore_errors=True)
            ok_count = sum(1 for r in new_rows if r["cache_ok"])
            elapsed = (time.time() - start_time) / 60
            print(f"{source_base} batch {batch_no}: cached {ok_count}/{len(new_rows)} cases, elapsed {elapsed:.1f} min")
            show_disk_usage(f"after cleanup {source_base} batch {batch_no}")
    finally:
        shutil.rmtree(STAGE_ROOT / source_base, ignore_errors=True)
        shutil.rmtree(BATCH_ROOT / source_base, ignore_errors=True)

cache_index = pd.DataFrame(cache_rows).drop_duplicates("case_id", keep="last")
cache_index.to_csv(cache_index_path, index=False, encoding="utf-8-sig")
clean.to_csv(CACHE_DIR / "q1_clean_manifest_for_cache.csv", index=False, encoding="utf-8-sig")

meta = {
    "image_size": IMAGE_SIZE,
    "channels": CHANNELS_25D,
    "n": n_total,
    "mmap_path": mmap_path.name,
    "source_manifest": str(manifest_path),
    "cache_case_limit": CACHE_CASE_LIMIT,
    "cache_only_splits": CACHE_ONLY_SPLITS,
    "batch_cases_per_extract": BATCH_CASES_PER_EXTRACT,
    "target_column": "model_target_text",
}
(CACHE_DIR / "q1_cache_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")

display(cache_index.groupby(["clean_split", "cache_ok"]).size().rename("cases").reset_index())
val_test = cache_index[cache_index["clean_split"].isin(["validation", "test"])]
if CACHE_ONLY_SPLITS is None or set(CACHE_ONLY_SPLITS) & {"validation", "test"}:
    assert len(val_test) > 0 and val_test["cache_ok"].all(), "Validation/test cache must be complete before final evaluation"
print("Saved cache:", CACHE_DIR)


In [ ]:
cache_index = pd.read_csv(CACHE_DIR / "q1_cache_index.csv")
ok_rows = cache_index[cache_index["cache_ok"] == True].head(4)
if len(ok_rows):
    fig, axes = plt.subplots(len(ok_rows), 6, figsize=(12, 2.2 * len(ok_rows)), constrained_layout=True)
    if len(ok_rows) == 1:
        axes = np.array([axes])
    titles = ["CT 35%", "CT 50%", "CT 65%", "PET axial MIP", "PET coronal MIP", "PET sagittal MIP"]
    for r, row in enumerate(ok_rows.itertuples(index=False)):
        img = arr[int(row.row_index)]
        for c in range(6):
            axes[r, c].imshow(img[c], cmap="gray" if c < 3 else "magma")
            axes[r, c].set_title(titles[c], fontsize=8)
            axes[r, c].axis("off")
        axes[r, 0].set_ylabel(str(row.case_id), fontsize=8)
    preview_path = CACHE_DIR / "q1_25d_cache_preview.png"
    fig.savefig(preview_path, dpi=170, bbox_inches="tight")
    plt.show()
    print("Preview:", preview_path)
